# GridAPPS-D Platform Connection

GridAPPS-D™ is an open-source platform that accelerates development and deployment of portable applications for advanced distribution management and operations. It is built in a Linux environment using Docker, which allows large software packages to be distributed as containers. Its purpose is to reduce the time and cost to integrate advanced functionality into distribution operations, to create a more reliable and resilient grid.

The GridAPPS-D source code is publicly available from the [GridAPPS-D GitHub](https://github.com/GRIDAPPSD). The GridAPPS-D™ project is sponsored by the U.S. Department of Energy Office of Electricity and receives ongoing updates from a team of core developers at PNNL. Documentation for the platform and API is available on the [GridAPPS-D Training ReadTheDocs](https://gridappsd-training.readthedocs.io/).

`GridappsdConnection` reads CIM models through the platform's `powergridmodel` API on the GOSS message bus, which runs SPARQL queries on the platform's [Blazegraph](3_3_blazegraph.ipynb) database.

## Installation and Setup

`GridappsdConnection` needs `gridappsd-python`, from the `gridappsd` extra:

```bash
pip install "cim-graph[gridappsd]"
```

The platform is started with [gridappsd-docker](https://github.com/GRIDAPPSD/gridappsd-docker), not with this repository's `docker-compose.yml`. Its Blazegraph also uses port 8889, so stop the docker compose `blazegraph` service first.

The examples need a running platform, so the outputs are not shown on this page.

## Connection Parameters

GridAPPS-D authenticates users on the GOSS message bus with a username and password. The default users and passwords are listed in the [GOSS Security Config File](https://github.com/GRIDAPPSD/GOSS-GridAPPS-D/blob/master/gov.pnnl.goss.gridappsd/conf/pnnl.goss.core.security.userfile.cfg).

The connection is configured with environment variables, read when it is created (see [Environment Variables](3_2_env_variables.ipynb)):

| Variable | Description |
|---|---|
| `CIMG_CIM_PROFILE` | `cimhub_2023` for GridAPPS-D releases v2025.01.0 and later, `rc4_2021` for v2024.09.0 and earlier |
| `CIMG_IEC61970_552` | Identifier style: `552-NEW` for v2025.01.0 and later, `552-LEGACY` for v2024.09.0 and earlier |
| `CIMG_HOST` | IP address of the platform, default `localhost` |
| `CIMG_PORT` | Port of the platform message bus, default `61613` |
| `CIMG_USERNAME` | Username from the GOSS security config (no default) |
| `CIMG_PASSWORD` | Password from the GOSS security config (no default) |
| `CIMG_DATABASE` | GridAPPS-D API used for queries, default `powergridmodel` (the only one supported) |

For a local platform, v2025.01.0 or later, with the default `system` user:

In [ ]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
os.environ['CIMG_IEC61970_552'] = '552-NEW'
os.environ['CIMG_HOST'] = 'localhost'
os.environ['CIMG_PORT'] = '61613'
os.environ['CIMG_USERNAME'] = 'system'
os.environ['CIMG_PASSWORD'] = 'manager'
os.environ['CIMG_DATABASE'] = 'powergridmodel'

import cimprofiles.cim17.cimhub_2023 as cim

## Creating a Connection

The connection logs in to the message bus on the first query.

In [ ]:
from cimgraph.databases import GridappsdConnection

gapps = GridappsdConnection()

## Loading a Feeder Model

`FeederModel` loads all equipment in the feeder with their terminals and connectivity nodes, and `get_all_edges` adds the attributes and associated objects of a class. `GridappsdConnection` ignores `CIMG_USE_UNITS`: unit values are plain floats in SI base units (here meters).

In [ ]:
from cimgraph.models import FeederModel

# IEEE 13-bus feeder
feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=gapps)

network.get_all_edges(cim.ACLineSegment)
line = network.first(cim.ACLineSegment)
print(f"Line {line.name}: length {line.length} m")

In [ ]:
gapps.disconnect()